# AuraGateway preflight-v3 exact-runtime resolution reconnaissance v1

Resolve the exact vLLM 0.25.1+cu129 / torch 2.11.0+cu129 dependency closure without package installation, GPU use, model loading, benchmark trajectories, credentials, or customer data.


In [ ]:
from __future__ import annotations

import hashlib
import importlib.metadata
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.parse
import urllib.request
from collections import Counter
from pathlib import Path, PurePosixPath

NOTEBOOK_NAME = "auragateway-preflight-v3-exact-runtime-resolution-reconnaissance-v1"
OUTPUT_DIRECTORY_NAME = (
    "auragateway_preflight_v3_exact_runtime_resolution_reconnaissance_v1"
)
OUTPUT_ROOT = Path("/kaggle/working") / OUTPUT_DIRECTORY_NAME
OUTPUT_ZIP = Path("/kaggle/working") / f"{OUTPUT_DIRECTORY_NAME}.zip"

EXPECTED_PYTHON = (3, 12)
EXPECTED_CUDA = "12.9"
EXPECTED_VLLM_DISTRIBUTION = "0.25.1+cu129"
EXPECTED_VLLM_WHEEL_SHA256 = (
    "9e206f370c934a2d4b6b1f05d3d09708d344e05d80260189ef19f60755709431"
)
EXPECTED_TORCH_VERSION = "2.11.0+cu129"

VLLM_RELEASE_TAG = "v0.25.1"
VLLM_RELEASE_API = (
    "https://api.github.com/repos/vllm-project/vllm/releases/tags/v0.25.1"
)
PYPI_INDEX = "https://pypi.org/simple"
PYTORCH_INDEX = "https://download.pytorch.org/whl/cu129"
NVIDIA_INDEX = "https://pypi.nvidia.com"

CREDENTIAL_ENV_NAMES = (
    "ANTHROPIC_API_KEY",
    "AWS_ACCESS_KEY_ID",
    "AWS_SECRET_ACCESS_KEY",
    "GOOGLE_API_KEY",
    "HF_TOKEN",
    "HUGGING_FACE_HUB_TOKEN",
    "OPENAI_API_KEY",
    "OPENROUTER_API_KEY",
)

REQUIRED_OUTPUTS = (
    "resolved_artifacts.json",
    "resolver_report.json",
    "host_policy.json",
    "resolution_receipt.json",
    "output_manifest.json",
)

SHA256_PATTERN = re.compile(r"^[0-9a-f]{64}$")


def canonical_json(payload: object) -> str:
    return json.dumps(
        payload,
        ensure_ascii=True,
        separators=(",", ":"),
        sort_keys=True,
    )


def sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def sha256_text(payload: str) -> str:
    return sha256_bytes(payload.encode("utf-8"))


def sha256_file(path: Path) -> str:
    return sha256_bytes(path.read_bytes())


def normalize_name(value: str) -> str:
    return re.sub(r"[-_.]+", "-", value.strip().lower())[:160]


def installed_snapshot() -> tuple[tuple[str, str], ...]:
    rows: set[tuple[str, str]] = set()
    for distribution in importlib.metadata.distributions():
        name = distribution.metadata.get("Name")
        version = distribution.version
        if isinstance(name, str) and isinstance(version, str):
            rows.add((normalize_name(name), version))
    return tuple(sorted(rows))


def archive_sha256(archive_info: object) -> str | None:
    if not isinstance(archive_info, dict):
        return None
    hashes = archive_info.get("hashes")
    if isinstance(hashes, dict):
        value = hashes.get("sha256")
        if isinstance(value, str) and SHA256_PATTERN.fullmatch(value):
            return value
    value = archive_info.get("hash")
    if isinstance(value, str) and value.startswith("sha256="):
        digest = value.removeprefix("sha256=")
        if SHA256_PATTERN.fullmatch(digest):
            return digest
    return None


def read_release_asset() -> dict[str, object]:
    request = urllib.request.Request(
        VLLM_RELEASE_API,
        headers={
            "Accept": "application/vnd.github+json",
            "User-Agent": "auragateway-exact-runtime-reconnaissance-v1",
        },
    )
    with urllib.request.urlopen(request, timeout=60) as response:
        payload = json.loads(response.read().decode("utf-8"))

    if not isinstance(payload, dict):
        raise RuntimeError("VLLM_RELEASE_RESPONSE_INVALID")
    assets = payload.get("assets")
    if not isinstance(assets, list):
        raise RuntimeError("VLLM_RELEASE_ASSETS_MISSING")

    candidates: list[dict[str, object]] = []
    for raw in assets:
        if not isinstance(raw, dict):
            continue
        name = raw.get("name")
        url = raw.get("browser_download_url")
        if not isinstance(name, str) or not isinstance(url, str):
            continue
        lowered = name.lower()
        if (
            lowered.endswith(".whl")
            and "0.25.1" in lowered
            and "cu129" in lowered
            and "x86_64" in lowered
        ):
            candidates.append(raw)

    if len(candidates) != 1:
        raise RuntimeError(
            f"VLLM_RELEASE_ASSET_CARDINALITY_INVALID:{len(candidates)}"
        )

    asset = candidates[0]
    name = asset["name"]
    url = asset["browser_download_url"]
    assert isinstance(name, str)
    assert isinstance(url, str)

    parsed = urllib.parse.urlsplit(url)
    if parsed.scheme != "https":
        raise RuntimeError("VLLM_RELEASE_ASSET_SCHEME_INVALID")
    if parsed.username is not None or parsed.password is not None:
        raise RuntimeError("VLLM_RELEASE_ASSET_CREDENTIALS_PRESENT")
    if parsed.query or parsed.fragment:
        raise RuntimeError("VLLM_RELEASE_ASSET_URL_NOT_STABLE")

    return {
        "release_tag": VLLM_RELEASE_TAG,
        "asset_name": name,
        "browser_download_url": url,
        "github_asset_id": asset.get("id"),
        "github_asset_size_bytes": asset.get("size"),
        "github_asset_digest": asset.get("digest"),
    }


def classify_authority(hostname: str) -> str:
    if hostname in {"download.pytorch.org", "download-r2.pytorch.org"}:
        return "pytorch_index"
    if hostname == "files.pythonhosted.org":
        return "pypi_files"
    if hostname in {
        "github.com",
        "objects.githubusercontent.com",
        "release-assets.githubusercontent.com",
    }:
        return "github_release"
    if hostname == "pypi.nvidia.com":
        return "nvidia_package_index"
    return "unclassified"


def evaluate_install_record(
    raw: object,
    *,
    index: int,
) -> dict[str, object]:
    if not isinstance(raw, dict):
        raise RuntimeError(f"RESOLUTION_RECORD_INVALID:{index}")
    metadata = raw.get("metadata")
    download_info = raw.get("download_info")
    if not isinstance(metadata, dict) or not isinstance(download_info, dict):
        raise RuntimeError(f"RESOLUTION_RECORD_SHAPE_INVALID:{index}")

    name = metadata.get("name")
    version = metadata.get("version")
    url = download_info.get("url")
    archive_info = download_info.get("archive_info")
    if not isinstance(name, str) or not isinstance(version, str):
        raise RuntimeError(f"RESOLUTION_METADATA_INVALID:{index}")
    if not isinstance(url, str):
        raise RuntimeError(f"RESOLUTION_URL_INVALID:{index}")

    parsed = urllib.parse.urlsplit(url)
    if parsed.scheme != "https":
        raise RuntimeError(f"RESOLUTION_URL_SCHEME_INVALID:{index}")
    if parsed.username is not None or parsed.password is not None:
        raise RuntimeError(f"RESOLUTION_URL_CREDENTIALS_PRESENT:{index}")
    query_present = bool(parsed.query)
    fragment_present = bool(parsed.fragment)

    hostname = parsed.hostname
    if not isinstance(hostname, str) or not hostname:
        raise RuntimeError(f"RESOLUTION_HOST_MISSING:{index}")

    artifact_filename = urllib.parse.unquote(
        PurePosixPath(parsed.path).name
    )
    if not artifact_filename.lower().endswith(".whl"):
        raise RuntimeError(f"NON_WHEEL_ARTIFACT_RESOLVED:{index}")

    digest = archive_sha256(archive_info)
    if digest is None:
        raise RuntimeError(f"RESOLUTION_SHA256_MISSING:{index}")

    sanitized_url = urllib.parse.urlunsplit(
        (parsed.scheme, hostname, parsed.path, "", "")
    )
    return {
        "record_index": index,
        "distribution_name": name,
        "normalized_name": normalize_name(name),
        "version": version,
        "artifact_filename": artifact_filename,
        "hostname": hostname,
        "sanitized_url": sanitized_url,
        "stable_url_sha256": sha256_text(sanitized_url),
        "query_present": query_present,
        "fragment_present": fragment_present,
        "sha256": digest,
        "source_authority": classify_authority(hostname),
    }


def write_json(path: Path, payload: object) -> None:
    path.write_text(
        canonical_json(payload) + "\n",
        encoding="utf-8",
        newline="\n",
    )


def write_failure(
    *,
    code: str,
    detail: str,
    stdout_sha256: str | None = None,
    stderr_sha256: str | None = None,
) -> None:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    write_json(
        OUTPUT_ROOT / "resolution_failure.json",
        {
            "schema_version": "1.0.0",
            "status": "FAILED_CLOSED",
            "failure_code": code,
            "detail": detail[:500],
            "stdout_sha256": stdout_sha256,
            "stderr_sha256": stderr_sha256,
            "package_installation_performed": False,
            "artifact_download_retention_permitted": False,
            "model_loads_performed": 0,
            "model_requests_performed": 0,
            "benchmark_trajectories_performed": 0,
            "credentials_used": False,
            "customer_data_used": False,
            "external_spend": 0,
        },
    )


if sys.version_info[:2] != EXPECTED_PYTHON:
    raise RuntimeError(
        f"PYTHON_VERSION_MISMATCH:{sys.version_info.major}."
        f"{sys.version_info.minor}"
    )

if OUTPUT_ROOT.exists() or OUTPUT_ZIP.exists():
    raise RuntimeError("OUTPUT_ALREADY_EXISTS")

input_root = Path("/kaggle/input")
if input_root.exists() and any(input_root.iterdir()):
    raise RuntimeError("KAGGLE_INPUTS_PRESENT")

present_credentials = [
    name for name in CREDENTIAL_ENV_NAMES if os.environ.get(name)
]
if present_credentials:
    raise RuntimeError(
        f"CREDENTIAL_ENV_PRESENT:count={len(present_credentials)}"
    )

before_packages = installed_snapshot()
release_asset = read_release_asset()
asset_url = release_asset["browser_download_url"]
assert isinstance(asset_url, str)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)

resolver_stdout_sha256: str | None = None
resolver_stderr_sha256: str | None = None

try:
    with tempfile.TemporaryDirectory(
        prefix="auragateway-preflight-v3-resolution-"
    ) as temporary_directory:
        temporary_root = Path(temporary_directory)
        report_path = temporary_root / "pip-resolution-report.json"
        cache_path = temporary_root / "pip-cache"
        cache_path.mkdir(parents=True, exist_ok=True)

        command = [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--dry-run",
            "--ignore-installed",
            "--only-binary=:all:",
            "--report",
            str(report_path),
            "--no-cache-dir",
            "--index-url",
            PYPI_INDEX,
            "--extra-index-url",
            PYTORCH_INDEX,
            "--extra-index-url",
            NVIDIA_INDEX,
            asset_url,
            f"torch=={EXPECTED_TORCH_VERSION}",
        ]
        command_contract = [
            "<python>",
            "-m",
            "pip",
            "install",
            "--dry-run",
            "--ignore-installed",
            "--only-binary=:all:",
            "--report",
            "<temporary-report>",
            "--no-cache-dir",
            "--index-url",
            PYPI_INDEX,
            "--extra-index-url",
            PYTORCH_INDEX,
            "--extra-index-url",
            NVIDIA_INDEX,
            "<exact-vllm-release-asset>",
            f"torch=={EXPECTED_TORCH_VERSION}",
        ]

        resolver_env = os.environ.copy()
        resolver_env["PIP_NO_CACHE_DIR"] = "1"
        resolver_env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"
        resolver_env["PIP_CACHE_DIR"] = str(cache_path)

        completed = subprocess.run(
            command,
            check=False,
            capture_output=True,
            text=True,
            timeout=1800,
            env=resolver_env,
        )
        resolver_stdout_sha256 = sha256_text(completed.stdout)
        resolver_stderr_sha256 = sha256_text(completed.stderr)

        if completed.returncode != 0:
            write_failure(
                code="PIP_RESOLUTION_FAILED",
                detail=f"returncode={completed.returncode}",
                stdout_sha256=resolver_stdout_sha256,
                stderr_sha256=resolver_stderr_sha256,
            )
            print(completed.stdout[-4000:])
            print(completed.stderr[-4000:], file=sys.stderr)
            raise RuntimeError("PIP_RESOLUTION_FAILED")

        if not report_path.is_file():
            raise RuntimeError("PIP_RESOLUTION_REPORT_MISSING")

        pip_report = json.loads(report_path.read_text(encoding="utf-8"))
        if not isinstance(pip_report, dict):
            raise RuntimeError("PIP_RESOLUTION_REPORT_INVALID")
        install_records = pip_report.get("install")
        if not isinstance(install_records, list) or not install_records:
            raise RuntimeError("PIP_RESOLUTION_INSTALL_SET_EMPTY")

        records = [
            evaluate_install_record(raw, index=index)
            for index, raw in enumerate(install_records)
        ]
        pip_version = pip_report.get("pip_version")
        if not isinstance(pip_version, str):
            pip_version = "unknown"

        temporary_wheels = tuple(temporary_root.rglob("*.whl"))
        transient_wheel_count = len(temporary_wheels)

    after_packages = installed_snapshot()
    if after_packages != before_packages:
        raise RuntimeError("PACKAGE_ENVIRONMENT_MUTATED")

    names = [record["normalized_name"] for record in records]
    if len(names) != len(set(names)):
        raise RuntimeError("RESOLUTION_DUPLICATE_DISTRIBUTION")

    by_name = {
        str(record["normalized_name"]): record
        for record in records
    }
    vllm_record = by_name.get("vllm")
    if vllm_record is None:
        raise RuntimeError("VLLM_RESOLUTION_RECORD_MISSING")
    if vllm_record["version"] != EXPECTED_VLLM_DISTRIBUTION:
        raise RuntimeError("VLLM_VERSION_MISMATCH")
    if vllm_record["sha256"] != EXPECTED_VLLM_WHEEL_SHA256:
        raise RuntimeError("VLLM_SHA256_MISMATCH")

    torch_record = by_name.get("torch")
    if torch_record is None:
        raise RuntimeError("TORCH_RESOLUTION_RECORD_MISSING")
    if torch_record["version"] != EXPECTED_TORCH_VERSION:
        raise RuntimeError("TORCH_VERSION_MISMATCH")

    host_counts = Counter(str(record["hostname"]) for record in records)
    hosts = [
        {
            "hostname": hostname,
            "distribution_count": count,
            "decision": "PENDING_REVIEW_FOR_EXACT_LOCK_FREEZE",
        }
        for hostname, count in sorted(host_counts.items())
    ]

    resolved_artifacts = {
        "schema_version": "1.0.0",
        "reconnaissance_id": NOTEBOOK_NAME,
        "runtime": {
            "python": "3.12",
            "cuda_variant": "cu129",
            "vllm_distribution": EXPECTED_VLLM_DISTRIBUTION,
            "vllm_wheel_sha256": EXPECTED_VLLM_WHEEL_SHA256,
            "torch": EXPECTED_TORCH_VERSION,
        },
        "record_count": len(records),
        "records": records,
    }

    resolver_report = {
        "schema_version": "1.0.0",
        "reconnaissance_id": NOTEBOOK_NAME,
        "python_version": platform.python_version(),
        "platform": platform.platform()[:240],
        "pip_version": pip_version,
        "pip_command_contract_sha256": sha256_text(
            canonical_json(command_contract)
        ),
        "pip_returncode": 0,
        "pip_stdout_sha256": resolver_stdout_sha256,
        "pip_stderr_sha256": resolver_stderr_sha256,
        "resolved_distribution_count": len(records),
        "host_count": len(host_counts),
        "transient_wheel_file_count_before_cleanup": transient_wheel_count,
        "temporary_resolution_directory_cleaned": True,
        "package_installation_performed": False,
        "artifact_download_retention_permitted": False,
        "model_loads_performed": 0,
        "model_requests_performed": 0,
        "benchmark_trajectories_performed": 0,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
    }

    host_policy = {
        "schema_version": "1.0.0",
        "policy_id": (
            "auragateway-preflight-v3-exact-runtime-resolution-host-review-v1"
        ),
        "review_status": "PENDING_REPOSITORY_REVIEW",
        "wildcard_hosts_permitted": False,
        "host_count": len(hosts),
        "hosts": hosts,
    }

    resolution_receipt = {
        "schema_version": "1.0.0",
        "receipt_id": (
            "auragateway-preflight-v3-exact-runtime-resolution-receipt-v1"
        ),
        "status": "COMPLETED_PENDING_REVIEW",
        "release_asset": {
            "release_tag": release_asset["release_tag"],
            "asset_name": release_asset["asset_name"],
            "github_asset_id": release_asset["github_asset_id"],
            "github_asset_size_bytes": release_asset[
                "github_asset_size_bytes"
            ],
            "github_asset_digest": release_asset["github_asset_digest"],
        },
        "exact_planned_vllm_identity_resolved": True,
        "exact_planned_vllm_sha256_matches_preflight_v3": True,
        "torch_2_11_0_cu129_resolved": True,
        "all_artifacts_have_sha256": all(
            isinstance(record["sha256"], str)
            and SHA256_PATTERN.fullmatch(str(record["sha256"])) is not None
            for record in records
        ),
        "all_artifact_hosts_explicitly_enumerated": True,
        "no_wildcard_hosts": True,
        "package_installation_performed": False,
        "artifact_download_retention_permitted": False,
        "retained_wheel_file_count": len(tuple(OUTPUT_ROOT.rglob("*.whl"))),
        "model_loads_performed": 0,
        "model_requests_performed": 0,
        "benchmark_trajectories_performed": 0,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
        "qualification_claimed": False,
        "exact_resolution_lock_frozen": False,
    }

    if resolution_receipt["retained_wheel_file_count"] != 0:
        raise RuntimeError("WHEEL_ARTIFACT_RETENTION_DETECTED")

    write_json(
        OUTPUT_ROOT / "resolved_artifacts.json",
        resolved_artifacts,
    )
    write_json(
        OUTPUT_ROOT / "resolver_report.json",
        resolver_report,
    )
    write_json(
        OUTPUT_ROOT / "host_policy.json",
        host_policy,
    )
    write_json(
        OUTPUT_ROOT / "resolution_receipt.json",
        resolution_receipt,
    )

    manifest_members = []
    for filename in REQUIRED_OUTPUTS[:-1]:
        path = OUTPUT_ROOT / filename
        manifest_members.append(
            {
                "path": filename,
                "sha256": sha256_file(path),
                "size_bytes": path.stat().st_size,
            }
        )

    output_manifest = {
        "schema_version": "1.0.0",
        "manifest_id": (
            "auragateway-preflight-v3-exact-runtime-resolution-output-manifest-v1"
        ),
        "member_count": len(manifest_members),
        "members": manifest_members,
        "qualification_claimed": False,
        "exact_resolution_lock_frozen": False,
    }
    write_json(
        OUTPUT_ROOT / "output_manifest.json",
        output_manifest,
    )

    missing_outputs = [
        filename
        for filename in REQUIRED_OUTPUTS
        if not (OUTPUT_ROOT / filename).is_file()
    ]
    if missing_outputs:
        raise RuntimeError(
            f"REQUIRED_OUTPUTS_MISSING:{missing_outputs}"
        )

    shutil.make_archive(
        str(OUTPUT_ZIP.with_suffix("")),
        "zip",
        root_dir=OUTPUT_ROOT,
    )

    summary = {
        "status": "COMPLETED_PENDING_REVIEW",
        "notebook_name": NOTEBOOK_NAME,
        "output_directory": OUTPUT_DIRECTORY_NAME,
        "output_zip": OUTPUT_ZIP.name,
        "resolved_distribution_count": len(records),
        "host_count": len(hosts),
        "exact_planned_vllm_sha256_matches_preflight_v3": True,
        "torch_2_11_0_cu129_resolved": True,
        "package_installation_performed": False,
        "artifact_download_retention_permitted": False,
        "retained_wheel_file_count": 0,
        "model_loads_performed": 0,
        "model_requests_performed": 0,
        "benchmark_trajectories_performed": 0,
        "credentials_used": False,
        "customer_data_used": False,
        "external_spend": 0,
        "qualification_claimed": False,
        "exact_resolution_lock_frozen": False,
        "save_this_notebook_output": True,
    }
    print(canonical_json(summary))
except Exception as error:
    if not (OUTPUT_ROOT / "resolution_failure.json").is_file():
        write_failure(
            code=type(error).__name__,
            detail=str(error),
            stdout_sha256=resolver_stdout_sha256,
            stderr_sha256=resolver_stderr_sha256,
        )
    raise
